# 🪟 PyQt5 – Lekce 5: Víceokenní aplikace a OOP

Zatím jsme psali veškerý kód lineárně do jednoho bloku. Ale co když chceme:
- otevřít **dialog** (nastavení, potvrzení, přihlášení),
- mít **více typů oken** (každé s jinou funkcionalitou),
- opakovaně vytvářet okna (seznam kontaktů → detail jednoho)?

Pro tyto případy přecházíme na **Objektově Orientované Programování (OOP)**.

---

## ⚙️ Proč OOP?

```
Bez OOP:
    Jeden velký blok kódu pro okno A + okno B
    → Kód se prolíná, nelze okno znovupoužít

S OOP:
    class OknoA(QWidget): ...
    class OknoB(QWidget): ...
    → Každé okno je samostatná "šablona", lze vytvořit kolikrát chceme
```

---

## ⚠️ Důležité: Garbage Collector

Pokud uložíte nové okno do **lokální proměnné**, Python ho po skončení funkce
okamžitě z paměti smaže – okno zmizí!

```python
def otevri():
    okno = NoveOkno()   # ❌ ŠPATNĚ – zmizí hned
    okno.show()

def otevri(self):
    self.okno = NoveOkno()  # ✅ SPRÁVNĚ – přidáno na instanci, přežije
    self.okno.show()
```


---

## 🟢 Ukázka 1 – Hlavní okno otevírá vedlejší okno

Základní vzor: jedno tlačítko v hlavním okně → otevře nové okno.

In [ ]:
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel

class VedlejsiOkno(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Vedlejší okno")
        self.resize(250, 120)
        layout = QVBoxLayout()
        layout.addWidget(QLabel("Toto je vedlejší okno!"))
        layout.addWidget(QPushButton("Toto tlačítko nic nedělá"))
        self.setLayout(layout)


class HlavniOkno(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Hlavní okno")
        self.resize(300, 100)
        layout = QVBoxLayout()
        self.tlacitko = QPushButton("Otevři vedlejší okno")
        self.tlacitko.clicked.connect(self.otevri_vedlejsi)
        layout.addWidget(self.tlacitko)
        self.setLayout(layout)

    def otevri_vedlejsi(self):
        self.vedlejsi = VedlejsiOkno()   # ← self., aby ho GC nesmazal!
        self.vedlejsi.show()


app = QApplication([])
okno = HlavniOkno()
okno.show()
app.exec()


---

## 🟡 Ukázka 2 – Předávání dat mezi okny

Hlavní okno předá vedlejšímu oknu data přes `__init__` parametr.

In [ ]:
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QLineEdit

class DetailOkno(QWidget):
    def __init__(self, jmeno):
        super().__init__()
        self.setWindowTitle(f"Detail uživatele: {jmeno}")
        self.resize(300, 150)
        layout = QVBoxLayout()
        layout.addWidget(QLabel(f"Jméno: {jmeno}"))
        layout.addWidget(QLabel("Role: Student"))
        layout.addWidget(QPushButton("Zavřít"))
        self.setLayout(layout)


class HlavniOkno(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Seznam uživatelů")
        self.resize(320, 100)
        layout = QHBoxLayout()

        self.vstup = QLineEdit()
        self.vstup.setPlaceholderText("Zadejte jméno…")
        self.btn = QPushButton("Zobraz detail")
        self.btn.clicked.connect(self.zobraz_detail)

        layout.addWidget(self.vstup)
        layout.addWidget(self.btn)
        self.setLayout(layout)

    def zobraz_detail(self):
        jmeno = self.vstup.text() or "Neznámý"
        self.detail = DetailOkno(jmeno)
        self.detail.show()


app = QApplication([])
okno = HlavniOkno()
okno.show()
app.exec()


---

## 🟡 Ukázka 3 – Modální dialog (okno blokující hlavní)

Modální okno (`exec()` místo `show()`) zablokuje hlavní okno, dokud ho uživatel nezavře.

In [ ]:
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QPushButton, QLabel, QDialog

class PotvrzeniDialog(QDialog):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Potvrzení")
        self.resize(260, 100)
        layout = QVBoxLayout()
        layout.addWidget(QLabel("Opravdu chcete akci provést?"))

        btn_ano = QPushButton("Ano")
        btn_ne = QPushButton("Ne")
        btn_ano.clicked.connect(self.accept)   # .accept() vrátí Accepted
        btn_ne.clicked.connect(self.reject)    # .reject() vrátí Rejected
        layout.addWidget(btn_ano)
        layout.addWidget(btn_ne)
        self.setLayout(layout)


class HlavniOkno(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Modální dialog")
        layout = QVBoxLayout()
        self.info = QLabel("Zatím žádná akce.")
        self.btn = QPushButton("Proveď akci")
        self.btn.clicked.connect(self.potvrdit_akci)
        layout.addWidget(self.info)
        layout.addWidget(self.btn)
        self.setLayout(layout)

    def potvrdit_akci(self):
        dialog = PotvrzeniDialog(self)
        if dialog.exec() == PotvrzeniDialog.Accepted:
            self.info.setText("✅ Akce provedena!")
        else:
            self.info.setText("❌ Akce zrušena.")


app = QApplication([])
okno = HlavniOkno()
okno.show()
app.exec()


---

# ✏️ Cvičení

---

## 🟢 Cvičení 1 – Dvě okna

Vytvořte aplikaci se dvěma tlačítky v hlavním okně:
- `"Otevři okno Sčítání"` → otevře okno pro sčítání dvou čísel,
- `"Otevři okno Násobení"` → otevře okno pro násobení dvou čísel.

Obě operační okna mají dvě vstupní pole a jedno tlačítko pro výpočet.

### Vaše řešení:

In [ ]:
from PyQt5.QtWidgets import QApplication, QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QLineEdit

# === ZDE PŘIJDE VÁŠ KÓD ===


# === KONEC VAŠEHO KÓDU ===


---

## 🔴 Cvičení 2 – Lékařská ordinace

Vytvořte aplikaci:
- **Hlavní okno:** obsahuje tlačítko `"Nová karta pacienta"` a seznam jmen (QLabel aktualizovaný po každém přidání).
- **Dialog:** modální, obsahuje vstup pro Jméno a Příjmení; tlačítko `"Uložit"` vrátí data hlavnímu oknu.

> 💡 Data předejte z dialogu přes vlastní metody nebo atributy objektu dialogu.

### Vaše řešení:

In [ ]:
from PyQt5.QtWidgets import QApplication, QWidget, QDialog, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QLineEdit, QFormLayout

# === ZDE PŘIJDE VÁŠ KÓD ===


# === KONEC VAŠEHO KÓDU ===
